# UV-Vis Spectroscopy and the Beer-Lambert Law

**Objective:** This lesson introduces one of the most common laboratory techniques for quantitative analysis: UV-Visible Spectroscopy. We will learn the theory behind the Beer-Lambert Law and apply it by creating and using a calibration curve to determine the concentration of an unknown sample.

**Learning Goals:**
1.  Understand the relationship between absorbance, concentration, and path length.
2.  State and apply the **Beer-Lambert Law**.
3.  Understand the purpose and construction of a **calibration curve**.
4.  Use **linear regression** (`SciPy`) to fit a line to experimental data and evaluate its quality ($R^2$).
5.  Use a derived calibration model to predict the concentration of an unknown sample.

## Part 1: The Theory - How UV-Vis Works

UV-Vis spectroscopy works by shining a beam of light through a sample and measuring how much of that light is absorbed. Molecules with certain structures (called chromophores) absorb light at specific wavelengths. The amount of light absorbed is directly proportional to the concentration of the molecule in the solution.

This relationship is quantified by the **Beer-Lambert Law**:
$$ A = \epsilon b C $$
where:
*   $A$ is the **Absorbance** (dimensionless), which is what the machine measures.
*   $\epsilon$ is the **molar absorptivity**, a constant specific to the molecule and wavelength (L/mol·cm).
*   $b$ is the **path length** of the cuvette holding the sample, almost always 1 cm.
*   $C$ is the **concentration** of the substance (mol/L).

Notice this is an equation for a straight line: $y = mx + c$, where $y=A$, $m=\epsilon b$, $x=C$, and the intercept $c=0$.

## Part 2: The Method - The Calibration Curve

While we could use the Beer-Lambert Law directly if we knew $\epsilon$, it's often more accurate and practical to create a **calibration curve**.

**The Workflow:**
1.  Prepare a series of standard solutions with known concentrations.
2.  Measure the absorbance of each standard solution at a fixed wavelength.
3.  Plot Absorbance (y-axis) vs. Concentration (x-axis).
4.  Perform a linear regression to find the best-fit line through the data points.
5.  Measure the absorbance of your unknown sample.
6.  Use the equation of your best-fit line to calculate the concentration of the unknown.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats # We'll use this for linear regression

# --- Part 3: Experimental Data ---
# Here is our simulated experimental data for our standard solutions.
# We've added a little random noise to make it realistic.
np.random.seed(42)
concentrations_mM = np.array([0, 2.0, 4.0, 6.0, 8.0, 10.0]) # in millimolar (mM)
absorbances = np.array([0.005, 0.168, 0.335, 0.501, 0.670, 0.832])
absorbances += np.random.normal(0, 0.01, size=absorbances.shape) # Add noise

# This is the sample we need to analyze
unknown_absorbance = 0.557

print("Experimental data is loaded.")

In [ ]:
# --- Part 4: Performing Linear Regression ---

# `scipy.stats.linregress` is a powerful function that does all the work for us.
# It returns several important values.
slope, intercept, r_value, p_value, std_err = stats.linregress(concentrations_mM, absorbances)

# The R-squared value tells us how well the line fits the data (1.0 is a perfect fit).
r_squared = r_value**2

print("Linear Regression Results:")
print(f"  Slope (εb): {slope:.4f}")
print(f"  Intercept:  {intercept:.4f}")
print(f"  R-squared:  {r_squared:.4f}")

# --- Plotting the Calibration Curve ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(10, 6))

# Plot the raw data points
plt.plot(concentrations_mM, absorbances, 'bo', label='Standard Data Points')

# Plot the best-fit line
fit_line = slope * concentrations_mM + intercept
plt.plot(concentrations_mM, fit_line, 'r-', label='Linear Fit')

# Add labels and annotations
plt.title('UV-Vis Calibration Curve', fontsize=16, weight='bold')
plt.xlabel('Concentration (mM)', fontsize=12)
plt.ylabel('Absorbance', fontsize=12)
plt.legend()
plt.grid(True)

# Add the equation and R^2 value to the plot
plt.text(1, 0.7, f'y = {slope:.4f}x + {intercept:.4f}\n$R^2$ = {r_squared:.4f}', fontsize=12)

plt.show()

## Part 5: Determining the Unknown Concentration

Now that we have our validated model (the best-fit line), we can use it to find the concentration of our unknown sample. We have the equation:
$$ A = (\text{slope}) \cdot C + (\text{intercept}) $$
We need to rearrange it to solve for $C$:
$$ C = \frac{A - \text{intercept}}{\text{slope}} $$

In [ ]:
# Calculate the concentration of the unknown sample
unknown_concentration = (unknown_absorbance - intercept) / slope

print(f"The measured absorbance of the unknown sample is {unknown_absorbance}.")
print(f"Using the calibration curve, the calculated concentration is {unknown_concentration:.3f} mM.")

# Let's add this to our plot to see where it falls
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(concentrations_mM, absorbances, 'bo', label='Standard Data Points')
ax.plot(concentrations_mM, slope * concentrations_mM + intercept, 'r-', label='Linear Fit')

# Highlight the unknown sample
ax.plot(unknown_concentration, unknown_absorbance, 'g*', markersize=15, label='Unknown Sample')

ax.set_title('Finding the Unknown on the Calibration Curve')
ax.set_xlabel('Concentration (mM)')
ax.set_ylabel('Absorbance')
ax.legend()
ax.grid(True)
plt.show()

## Student Challenges

1.  **Outliers:** What happens if one of your standard measurements was bad? Add a significant error to one of the `absorbances` in Part 3 (e.g., change `0.501` to `0.601`). Re-run the analysis. How does this outlier affect the R-squared value? How much does it change the final calculated concentration for the unknown? This demonstrates the importance of good lab technique.

2.  **Limits of the Model:** The Beer-Lambert Law is only linear for a certain range of concentrations (typically for A < 1.0). What would you do if you measured your unknown and its absorbance was `1.8`? Can you trust the result from your calibration curve? What would be the correct experimental procedure in this situation?